# 🩻 Shoulder X-ray Classification — Model Training

Train an **EfficientNetB0** model to classify shoulder X-rays as **NORMAL** or **ABNORMAL** using the MURA dataset.

**Two-phase training:**
1. Phase 1: Train custom head with frozen EfficientNet base
2. Phase 2: Fine-tune top layers of EfficientNet

> ⚠️ This notebook is for educational purposes only.

## 1. Setup & Mount Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!pip install -q tensorflow numpy matplotlib scikit-learn seaborn

## 2. Configuration

Update `DATA_DIR` below to point to your MURA dataset location in Google Drive.

In [ ]:
import os
import glob
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt

# ── Configuration ──
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
INITIAL_EPOCHS = 10
FINE_TUNE_EPOCHS = 10
LEARNING_RATE = 1e-3
FINE_TUNE_LR = 1e-5

# ⬇️ UPDATE THIS PATH to your MURA dataset location in Google Drive
DATA_DIR = '/content/drive/MyDrive/MURA-v1.1'
TRAIN_DIR = os.path.join(DATA_DIR, 'train', 'XR_SHOULDER')
VALID_DIR = os.path.join(DATA_DIR, 'valid', 'XR_SHOULDER')
MODEL_SAVE_PATH = '/content/shoulder_xray_model.h5'

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

## 3. Load & Explore Dataset

In [ ]:
def prepare_dataframe(root_dir):
    """Walk MURA shoulder directory and create (file_path, label) lists."""
    file_paths = []
    labels = []
    for patient_dir in sorted(glob.glob(os.path.join(root_dir, 'patient*'))):
        for study_dir in sorted(glob.glob(os.path.join(patient_dir, 'study*'))):
            label = 1 if 'positive' in study_dir.lower() else 0
            for img_path in glob.glob(os.path.join(study_dir, '*.png')):
                file_paths.append(img_path)
                labels.append(label)
    return file_paths, labels

# Load data
train_paths, train_labels = prepare_dataframe(TRAIN_DIR)
valid_paths, valid_labels = prepare_dataframe(VALID_DIR)

print(f'Training images:   {len(train_paths)}')
print(f'  Normal:          {train_labels.count(0)}')
print(f'  Abnormal:        {train_labels.count(1)}')
print(f'Validation images: {len(valid_paths)}')
print(f'  Normal:          {valid_labels.count(0)}')
print(f'  Abnormal:        {valid_labels.count(1)}')

## 4. Data Pipeline with Augmentation

**Key fix**: Uses `preprocess_input` from EfficientNet instead of simple `/255.0` normalization.

In [ ]:
def load_and_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    # EfficientNet expects float [0, 255] → its own normalization
    img = tf.cast(img, tf.float32)
    img = preprocess_input(img)
    return img, label

def augment_image(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=0.15)
    img = tf.image.random_contrast(img, lower=0.8, upper=1.2)
    img = tf.image.resize_with_crop_or_pad(img, 248, 248)
    img = tf.image.random_crop(img, size=[224, 224, 3])
    return img, label

# Create datasets
train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_ds = train_ds.shuffle(len(train_paths), seed=42)
train_ds = train_ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

valid_ds = tf.data.Dataset.from_tensor_slices((valid_paths, valid_labels))
valid_ds = valid_ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
valid_ds = valid_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print('✅ Data pipelines created with EfficientNet preprocessing.')

## 5. Visualize Sample Images

In [ ]:
plt.figure(figsize=(14, 6))
for images, labels in train_ds.take(1):
    for i in range(min(8, len(images))):
        plt.subplot(2, 4, i + 1)
        # Reverse preprocess_input for display
        display_img = images[i].numpy()
        display_img = (display_img - display_img.min()) / (display_img.max() - display_img.min())
        plt.imshow(display_img)
        label_text = 'ABNORMAL' if labels[i].numpy() == 1 else 'NORMAL'
        color = 'red' if labels[i].numpy() == 1 else 'green'
        plt.title(label_text, color=color, fontweight='bold')
        plt.axis('off')
plt.suptitle('Sample Training Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Build Model

In [ ]:
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3),
)
base_model.trainable = False  # Freeze base layers

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid'),
])

model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

model.summary()

## 7. Phase 1: Train Head (Frozen Base)

In [ ]:
callbacks = [
    ModelCheckpoint(MODEL_SAVE_PATH, monitor='val_accuracy', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1),
]

history = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=INITIAL_EPOCHS,
    callbacks=callbacks,
)

phase1_acc = max(history.history['val_accuracy'])
print(f'\n✅ Phase 1 best val_accuracy: {phase1_acc:.2%}')

## 8. Phase 2: Fine-Tune Top Layers

In [ ]:
# Unfreeze the base model
base_model.trainable = True

# Freeze all layers except the last 20
for layer in base_model.layers[:-20]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f'Unfrozen {trainable_count} of {len(base_model.layers)} base layers')

# Recompile with lower learning rate
model.compile(
    optimizer=Adam(learning_rate=FINE_TUNE_LR),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

callbacks_ft = [
    ModelCheckpoint(MODEL_SAVE_PATH, monitor='val_accuracy', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-8, verbose=1),
]

history_fine = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=callbacks_ft,
)

phase2_acc = max(history_fine.history['val_accuracy'])
print(f'\n✅ Phase 2 best val_accuracy: {phase2_acc:.2%}')
print(f'   (Phase 1 was: {phase1_acc:.2%})')

## 9. Training History Visualization

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Combine histories
all_acc = history.history['accuracy'] + history_fine.history['accuracy']
all_val_acc = history.history['val_accuracy'] + history_fine.history['val_accuracy']
all_loss = history.history['loss'] + history_fine.history['loss']
all_val_loss = history.history['val_loss'] + history_fine.history['val_loss']

# Accuracy
ax1.plot(all_acc, label='Train Accuracy', linewidth=2)
ax1.plot(all_val_acc, label='Val Accuracy', linewidth=2)
ax1.axvline(x=len(history.history['accuracy'])-1, color='gray', linestyle='--', label='Fine-tune start')
ax1.set_title('Model Accuracy', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss
ax2.plot(all_loss, label='Train Loss', linewidth=2)
ax2.plot(all_val_loss, label='Val Loss', linewidth=2)
ax2.axvline(x=len(history.history['loss'])-1, color='gray', linestyle='--', label='Fine-tune start')
ax2.set_title('Model Loss', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Training History (Phase 1 + Phase 2)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Evaluate on Validation Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

y_true, y_pred = [], []
for images, labels in valid_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend((preds[:, 0] >= 0.5).astype(int))

print('Classification Report:')
print('=' * 50)
print(classification_report(y_true, y_pred, target_names=['NORMAL', 'ABNORMAL']))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['NORMAL', 'ABNORMAL'],
            yticklabels=['NORMAL', 'ABNORMAL'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Save & Download Model

In [ ]:
drive_save_path = '/content/drive/MyDrive/shoulder_xray_model.h5'
model.save(drive_save_path)
print(f'✅ Model saved to Google Drive: {drive_save_path}')
print(f'✅ Model saved locally: {MODEL_SAVE_PATH}')
print('\n📥 Download and place in: backend/model.h5')

from google.colab import files
files.download(MODEL_SAVE_PATH)

## 12. Test Grad-CAM (Preview)

In [ ]:
import cv2

def generate_gradcam(model, img_array, layer_name=None):
    if layer_name is None:
        for layer in reversed(model.layers[0].layers):
            if isinstance(layer, tf.keras.layers.Conv2D):
                layer_name = layer.name
                break

    grad_model = tf.keras.models.Model(
        inputs=model.input,
        outputs=[model.layers[0].get_layer(layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, 0]
    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_outputs[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return cv2.resize(heatmap.numpy(), (224, 224))

# Test on a sample image
for images, labels in valid_ds.take(1):
    sample_img = images[0:1]
    sample_label = labels[0].numpy()
    break

pred = model.predict(sample_img, verbose=0)[0][0]
heatmap = generate_gradcam(model, sample_img)

# Reverse preprocess for display
img_display = sample_img[0].numpy()
img_display = ((img_display - img_display.min()) / (img_display.max() - img_display.min()) * 255).astype(np.uint8)
heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
overlay = np.uint8(0.4 * heatmap_colored + 0.6 * img_display)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(img_display)
axes[0].set_title('Original X-ray')
axes[0].axis('off')
axes[1].imshow(heatmap, cmap='jet')
axes[1].set_title('Grad-CAM Heatmap')
axes[1].axis('off')
axes[2].imshow(overlay)
axes[2].set_title(f'Overlay (Pred: {pred:.2f})')
axes[2].axis('off')
plt.suptitle(f'Actual: {"ABNORMAL" if sample_label == 1 else "NORMAL"}', fontweight='bold')
plt.tight_layout()
plt.show()